In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install libero


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 16.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.7/217.7 kB 19.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.9/192.9 kB 21.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 19.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.5/217.5 kB 24.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 90

## 1. GT Bounding Box & World Coordinate Batch Generation

In [ ]:
import h5py
import json
import numpy as np
import os
import time
import shutil
import glob
from robosuite.utils import camera_utils as CU
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

# ═══════════════════════════════════════════════════════════════════════════════
# 1. CONFIG & PATHS
# ═══════════════════════════════════════════════════════════════════════════════
INPUT_DIR = "/content/drive/MyDrive/LIBERO/libero_spatial/"
OUTPUT_DIR = "/content/output/"

TARGET_CAMERAS = ["agentview", "robot0_eye_in_hand"]

IMG_W, IMG_H = 128, 128
BBOX_PADDING = 5
EXCLUDE_PREFIXES = ("robot0", "gripper")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. GEOMETRY ENGINE
# ═══════════════════════════════════════════════════════════════════════════════
def get_all_geoms_for_body(env, body_name):
    root_id = env.sim.model.body_name2id(body_name)
    descendant_ids = set()
    for bid in range(env.sim.model.nbody):
        current = bid
        while current != 0:
            if current == root_id: descendant_ids.add(bid); break
            current = env.sim.model.body_parentid[current]
    return [gid for gid in range(env.sim.model.ngeom) if env.sim.model.geom_bodyid[gid] in descendant_ids]

def world_to_pixel(world_pos, extrinsic_inv, K, img_h, img_w):
    pt_cam = extrinsic_inv @ np.append(world_pos, 1.0)
    if pt_cam[2] <= 0.01: return None
    u = K[0, 0] * pt_cam[0] / pt_cam[2] + K[0, 2]
    v = K[1, 1] * pt_cam[1] / pt_cam[2] + K[1, 2]
    return int(u), int(img_h - v)

def get_bbox(env, body_name, extrinsic_inv, K, img_h, img_w, padding=BBOX_PADDING):
    geom_ids = get_all_geoms_for_body(env, body_name)
    if not geom_ids: return None
    pixels = []
    for gid in geom_ids:
        pos, size = env.sim.data.geom_xpos[gid], env.sim.model.geom_size[gid]
        xmat = env.sim.data.geom_xmat[gid].reshape(3, 3)
        for sx in [-1, 1]:
            for sy in [-1, 1]:
                for sz in [-1, 1]:
                    rotated_offset = xmat @ (np.array([sx, sy, sz]) * size)
                    px = world_to_pixel(pos + rotated_offset, extrinsic_inv, K, img_h, img_w)
                    if px is not None: pixels.append(px)
    if not pixels: return None
    us, vs = zip(*pixels)
    rx1, ry1, rx2, ry2 = min(us)-padding, min(vs)-padding, max(us)+padding, max(vs)+padding
    if (rx2-rx1)>(img_w*3) or (ry2-ry1)>(img_h*3): return None
    x1, y1, x2, y2 = max(0,int(rx1)), max(0,int(ry1)), min(img_w,int(rx2)), min(img_h,int(ry2))
    return (x1, y1, x2, y2) if x1 < x2 and y1 < y2 else None

def discover_objects(env):
    return {env.sim.model.body_id2name(i).replace("_main",""): env.sim.model.body_id2name(i)
           for i in range(env.sim.model.nbody) if env.sim.model.body_id2name(i).endswith("_main")
           and not any(env.sim.model.body_id2name(i).startswith(p) for p in EXCLUDE_PREFIXES)}

def check_coords_complete(f):
    """Check ALL demos have bboxes + world_coords for BOTH cameras."""
    demo_keys = list(f["data"].keys())
    required = [f"{cam}_{s}" for cam in TARGET_CAMERAS for s in ["bboxes", "world_coords"]]
    for demo_id in demo_keys:
        obs = f[f"data/{demo_id}/obs"]
        for key in required:
            if key not in obs:
                return False
    return True

# ═══════════════════════════════════════════════════════════════════════════════
# 3. BATCH LOOP
# ═══════════════════════════════════════════════════════════════════════════════
os.makedirs(OUTPUT_DIR, exist_ok=True)

HDF5_FILES = sorted(glob.glob(os.path.join(INPUT_DIR, "*.hdf5")))
benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict["libero_spatial"]()

if not HDF5_FILES:
    print(f"❌ NO HDF5 FILES FOUND in {INPUT_DIR}")
else:
    print(f"📂 INPUT:  {INPUT_DIR} ({len(HDF5_FILES)} files)")
    print(f"📂 OUTPUT: {OUTPUT_DIR}")
    print(f"📐 BBOX RESOLUTION: {IMG_W}×{IMG_H}")
    print("=" * 80)

total_files = len(HDF5_FILES)
skipped = 0
processed = 0
errors = 0

for file_idx, input_path in enumerate(HDF5_FILES):
    file_name = os.path.basename(input_path)
    output_path = os.path.join(OUTPUT_DIR, file_name)

    print(f"\n{'─' * 80}")
    print(f"[{file_idx+1}/{total_files}] 📄 {file_name}")

    # --- SKIP CHECK: output file exists and complete ---
    if os.path.exists(output_path):
        try:
            with h5py.File(output_path, "r") as f_check:
                if check_coords_complete(f_check):
                    n_demos = len(list(f_check["data"].keys()))
                    print(f"  ⏭️  SKIP: All coords complete ({n_demos} demos × 2 cameras)")
                    skipped += 1
                    continue
                else:
                    print(f"  ⚠️  Output incomplete — reprocessing")
        except Exception as e:
            print(f"  ⚠️  Output unreadable ({e}) — reprocessing")

    # --- COPY input → output ---
    print(f"  📋 Copying to output dir...")
    shutil.copy(input_path, output_path)

    # --- FIND TASK ---
    task_fn = file_name.replace("_demo.hdf5", "")
    task_idx = next(i for i in range(10) if (task_suite.get_task(i).name in task_fn) or (task_fn in task_suite.get_task(i).name))
    task = task_suite.get_task(task_idx)
    print(f"  🎯 Task {task_idx}: {task.name}")

    # --- CREATE ENV ---
    env = OffScreenRenderEnv(
        bddl_file_name=os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file),
        camera_heights=IMG_H, camera_widths=IMG_W
    )

    with h5py.File(output_path, "r+") as f:
        demo_keys = sorted(list(f["data"].keys()))
        n_demos = len(demo_keys)
        env.reset()
        objects = discover_objects(env)
        print(f"  🔍 Objects: {list(objects.keys())}")

        for camera_name in TARGET_CAMERAS:
            t_start = time.time()
            K = CU.get_camera_intrinsic_matrix(env.sim, camera_name, IMG_H, IMG_W)
            total_frames = 0

            for demo_idx, demo_id in enumerate(demo_keys):
                states = f[f"data/{demo_id}/states"][:]
                traj_pixel, traj_world = [], []

                for t in range(len(states)):
                    env.sim.set_state_from_flattened(states[t])
                    env.sim.forward()
                    ext_inv = np.linalg.inv(CU.get_camera_extrinsic_matrix(env.sim, camera_name))
                    f_px, f_wd = {}, {}
                    for label, body in objects.items():
                        bbox = get_bbox(env, body, ext_inv, K, IMG_H, IMG_W)
                        if bbox:
                            f_px[label] = bbox
                        bid = env.sim.model.body_name2id(body)
                        f_wd[label] = {
                            "pos": env.sim.data.body_xpos[bid].tolist(),
                            "mat": env.sim.data.body_xmat[bid].reshape(3, 3).tolist()
                        }
                    traj_pixel.append(f_px)
                    traj_world.append(f_wd)

                total_frames += len(states)

                obs = f[f"data/{demo_id}/obs"]
                for k, d in [(f"{camera_name}_bboxes", traj_pixel),
                             (f"{camera_name}_world_coords", traj_world)]:
                    if k in obs:
                        del obs[k]
                    obs.create_dataset(k, data=json.dumps(d))

                if (demo_idx + 1) % 10 == 0 or demo_idx == n_demos - 1:
                    print(f"    📊 {camera_name}: {demo_idx+1}/{n_demos} demos done...")

            elapsed = time.time() - t_start
            print(f"  ✅ {camera_name}: {total_frames} frames in {elapsed:.1f}s")

    env.close()
    print(f"  💾 SAVED: {output_path}")
    processed += 1

# ═══════════════════════════════════════════════════════════════════════════════
# 4. SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 80}")
print(f"🎉 BATCH COMPLETE")
print(f"   Processed: {processed} | Skipped: {skipped} | Errors: {errors} | Total: {total_files}")
print(f"   Output: {OUTPUT_DIR}")
print(f"{'=' * 80}")

## 2. Scene Graph Batch Generation (Agentview + Wrist)

In [ ]:
import h5py, json, numpy as np, os, time, glob, shutil

# ═══════════════════════════════════════════════════════════════════════════════
# 1. CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
INPUT_DIR  = "/content/output/"
OUTPUT_DIR = "/content/output_final/"
OVERWRITE_EXISTING_GRAPHS = True

TARGET_CAMERAS = ["agentview", "robot0_eye_in_hand"]
CONTAINMENT_THRESH = 0.8

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 2. SCENE GRAPH ENGINE
# ═══════════════════════════════════════════════════════════════════════════════
def generate_frame_graph(bboxes, world, object_filter=None, is_drawer_task=False):
    if object_filter is not None:
        objects = sorted([o for o in bboxes.keys() if o in object_filter])
    else:
        objects = sorted(list(bboxes.keys()))

    triplets = []

    for A in objects:
        if A not in world: continue
        pos_a = np.array(world[A]['pos'])
        for B in objects:
            if A == B: continue
            if B not in world: continue
            pos_b = np.array(world[B]['pos'])

            is_stacked = False
            if A in bboxes and B in bboxes:
                tx1,ty1,tx2,ty2 = bboxes[A]
                bx1,by1,bx2,by2 = bboxes[B]
                ix1,iy1 = max(tx1,bx1), max(ty1,by1)
                ix2,iy2 = min(tx2,bx2), min(ty2,by2)
                if ix2 > ix1 and iy2 > iy1:
                    inter = (ix2-ix1)*(iy2-iy1)
                    a_area = (tx2-tx1)*(ty2-ty1)
                    b_area = (bx2-bx1)*(by2-by1)
                    io_min = inter / min(a_area, b_area)
                    if io_min > CONTAINMENT_THRESH and ("bowl" in A or "bowl" in B):
                        is_stacked = True
                        pair = {A, B} == {"akita_black_bowl_1", "wooden_cabinet_1"}
                        if is_drawer_task and pair:
                            if pos_a[2] >= pos_b[2]:
                                triplets.append((A, "is_inside", B))
                            else:
                                triplets.append((A, "contains", B))
                        else:
                            if pos_a[2] >= pos_b[2]:
                                triplets.append((A, "is_on_top_of", B))
                            else:
                                triplets.append((A, "is_below_of", B))
            if is_stacked: continue

            dx = pos_a[0] - pos_b[0]
            dy = pos_a[1] - pos_b[1]
            if abs(dx) >= abs(dy):
                triplets.append((A, "is_in_front_of" if dx > 0 else "is_behind", B))
            else:
                triplets.append((A, "is_left_of" if dy > 0 else "is_right_of", B))

    return triplets


# ═══════════════════════════════════════════════════════════════════════════════
# 3. VALIDATION HELPERS
# ═══════════════════════════════════════════════════════════════════════════════
def check_bboxes_complete(f, demo_keys):
    required_keys = [f"{cam}_{suffix}"
                     for cam in TARGET_CAMERAS
                     for suffix in ["bboxes", "world_coords"]]
    missing = []
    for demo_id in demo_keys:
        obs = f[f"data/{demo_id}/obs"]
        for key in required_keys:
            if key not in obs:
                missing.append(f"{demo_id}/obs/{key}")
    return missing

def check_graphs_complete(f, demo_keys):
    graph_keys = [f"{cam}_scene_graph" for cam in TARGET_CAMERAS]
    missing = []
    for demo_id in demo_keys:
        obs = f[f"data/{demo_id}/obs"]
        for key in graph_keys:
            if key not in obs:
                missing.append(f"{demo_id}/obs/{key}")
    return missing


# ═══════════════════════════════════════════════════════════════════════════════
# 4. BATCH LOOP
# ═══════════════════════════════════════════════════════════════════════════════
HDF5_FILES = sorted(glob.glob(os.path.join(INPUT_DIR, "*.hdf5")))

if not HDF5_FILES:
    print(f"❌ NO HDF5 FILES FOUND in {INPUT_DIR}")
else:
    print(f"📂 Found {len(HDF5_FILES)} HDF5 files in {INPUT_DIR}")
    print(f"🔧 OVERWRITE_EXISTING_GRAPHS = {OVERWRITE_EXISTING_GRAPHS}")
    print(f"📁 Output dir: {OUTPUT_DIR}")
    print("=" * 80)

total_files = len(HDF5_FILES)
skipped = 0
processed = 0
errors = 0

for file_idx, hdf5_path in enumerate(HDF5_FILES):
    file_name = os.path.basename(hdf5_path)
    local_path = os.path.join(OUTPUT_DIR, file_name)
    is_drawer_task = "in_the_top_drawer" in file_name

    print(f"\n{'─' * 80}")
    print(f"[{file_idx+1}/{total_files}] 📄 {file_name}")
    if is_drawer_task:
        print(f"  🗄️  DRAWER TASK — is_inside override enabled")

    shutil.copy2(hdf5_path, local_path)
    print(f"  📥 Copied to local: {local_path}")

    with h5py.File(local_path, "r+") as f:
        demo_keys = sorted(list(f["data"].keys()))
        n_demos = len(demo_keys)

        bbox_missing = check_bboxes_complete(f, demo_keys)
        if bbox_missing:
            print(f"  ❌ BBOX DATA MISSING — cannot generate graphs. Missing keys:")
            for m in bbox_missing[:5]:
                print(f"     • {m}")
            if len(bbox_missing) > 5:
                print(f"     ... and {len(bbox_missing) - 5} more")
            errors += 1
            continue

        print(f"  ✅ Bboxes verified: {n_demos} demos × 2 cameras × 2 keys")

        graph_missing = check_graphs_complete(f, demo_keys)

        if not graph_missing and not OVERWRITE_EXISTING_GRAPHS:
            print(f"  ⏭️  SKIP: All graphs already present ({n_demos} demos × 2 cameras)")
            skipped += 1
            continue

        if not graph_missing and OVERWRITE_EXISTING_GRAPHS:
            print(f"  🔄 OVERWRITE MODE: Recomputing all graphs")
        elif graph_missing:
            print(f"  🔨 GENERATING: {len(graph_missing)} graph keys missing")

        t_start = time.time()
        total_triplets_agent = 0
        total_triplets_wrist = 0
        total_frames = 0
        total_inside_frames = 0

        for demo_idx, demo_id in enumerate(demo_keys):
            obs = f[f"data/{demo_id}/obs"]

            agent_bboxes_all = json.loads(obs["agentview_bboxes"][()].decode('utf-8'))
            agent_world_all  = json.loads(obs["agentview_world_coords"][()].decode('utf-8'))
            wrist_bboxes_all = json.loads(obs["robot0_eye_in_hand_bboxes"][()].decode('utf-8'))

            n_frames = len(agent_bboxes_all)
            total_frames += n_frames
            agent_graphs = []
            wrist_graphs = []

            for frame_idx in range(n_frames):
                agent_bboxes = agent_bboxes_all[frame_idx]
                agent_world  = agent_world_all[frame_idx]
                wrist_bboxes = wrist_bboxes_all[frame_idx]

                agent_triplets = generate_frame_graph(agent_bboxes, agent_world,
                                                      is_drawer_task=is_drawer_task)
                agent_graphs.append(agent_triplets)
                total_triplets_agent += len(agent_triplets)

                if is_drawer_task:
                    total_inside_frames += sum(1 for t in agent_triplets if t[1] == "is_inside")

                wrist_triplets = generate_frame_graph(agent_bboxes, agent_world,
                                                      object_filter=set(wrist_bboxes.keys()),
                                                      is_drawer_task=is_drawer_task)
                wrist_graphs.append(wrist_triplets)
                total_triplets_wrist += len(wrist_triplets)

            for key, data in [("agentview_scene_graph", agent_graphs),
                               ("robot0_eye_in_hand_scene_graph", wrist_graphs)]:
                if key in obs: del obs[key]
                obs.create_dataset(key, data=json.dumps(data))

            if (demo_idx + 1) % 10 == 0 or demo_idx == n_demos - 1:
                print(f"    📊 {demo_idx+1}/{n_demos} demos done...")

        elapsed = time.time() - t_start
        avg_agent = total_triplets_agent / total_frames if total_frames > 0 else 0
        avg_wrist = total_triplets_wrist / total_frames if total_frames > 0 else 0

        print(f"  ✅ DONE in {elapsed:.1f}s")
        print(f"     {total_frames} frames across {n_demos} demos")
        print(f"     Agentview: {total_triplets_agent} triplets (avg {avg_agent:.1f}/frame)")
        print(f"     Wrist:     {total_triplets_wrist} triplets (avg {avg_wrist:.1f}/frame)")
        if is_drawer_task:
            print(f"     is_inside frames: {total_inside_frames}")
        processed += 1

# ═══════════════════════════════════════════════════════════════════════════════
# 5. SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 80}")
print(f"🎉 BATCH COMPLETE")
print(f"   Processed: {processed} | Skipped: {skipped} | Errors: {errors} | Total: {total_files}")
print(f"   ➡️  Run Cell B to copy outputs to Drive.")
print(f"{'=' * 80}")